# Лабораторна робота №2
## Наука про дані: підготовчий етап
### Налаштування віртуального середовища та інсталяція бібліотек

In [99]:
# Створення віртуального середовища у терміналі (виконується одноразово)
# python -m venv venv
# Активація середовища:
# Windows: venv\Scripts\activate
# Linux/MacOS: source venv/bin/activate

# Інсталяція необхідних пакетів:
# pip install pandas requests

### Імпорт модулів та константні дані

In [100]:
import os
import re
import urllib.request
from datetime import datetime
import pandas as pd

NOAA_TO_UKR = {
    1: {"name": "Вінницька", "ukr_id": 1},
    2: {"name": "Волинська", "ukr_id": 2},
    3: {"name": "Дніпропетровська", "ukr_id": 3},
    4: {"name": "Донецька", "ukr_id": 4},
    5: {"name": "Житомирська", "ukr_id": 5},
    6: {"name": "Закарпатська", "ukr_id": 6},
    7: {"name": "Запорізька", "ukr_id": 7},
    8: {"name": "Івано-Франківська", "ukr_id": 8},
    9: {"name": "Київська", "ukr_id": 9},
    10: {"name": "Кіровоградська", "ukr_id": 10},
    11: {"name": "Луганська", "ukr_id": 11},
    12: {"name": "Львівська", "ukr_id": 12},
    13: {"name": "Миколаївська", "ukr_id": 13},
    14: {"name": "Одеська", "ukr_id": 14},
    15: {"name": "Полтавська", "ukr_id": 15},
    16: {"name": "Рівненська", "ukr_id": 16},
    17: {"name": "Сумська", "ukr_id": 17},
    18: {"name": "Тернопільська", "ukr_id": 18},
    19: {"name": "Харківська", "ukr_id": 19},
    20: {"name": "Херсонська", "ukr_id": 20},
    21: {"name": "Хмельницька", "ukr_id": 21},
    22: {"name": "Черкаська", "ukr_id": 22},
    23: {"name": "Чернівецька", "ukr_id": 23},
    24: {"name": "Чернігівська", "ukr_id": 24},
    25: {"name": "Крим", "ukr_id": 25},
    26: {"name": "Київ", "ukr_id": 26},
    27: {"name": "Севастополь", "ukr_id": 27}
}

DOWNLOAD_DIR = "vhi_data"

### Завдання: 
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [101]:
import shutil
if os.path.exists("vhi_data"):
    shutil.rmtree("vhi_data")
    print("Стару папку зі старим сміттям успішно видалено!")

In [102]:
def download_vhi_data(directory=DOWNLOAD_DIR):
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Створено директорію: {directory}")
    
    existing_files = os.listdir(directory)
    if existing_files:
        print("Директорія не порожня. Запобігання повторному довантаженню активовано. Завантаження скасовано.")
        return

    base_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={pid}&year1=1981&year2=2024&type=Mean"
    
    for pid in range(1, 28):
        url = base_url.format(pid=pid)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        # Змінили .txt на .csv за коментарем Антона Олеговича
        filename = f"vhi_id_{pid}_{timestamp}.csv"
        filepath = os.path.join(directory, filename)
        
        try:
            print(f"Завантаження з підключенням до ID {pid}...")
            urllib.request.urlretrieve(url, filepath)
        except Exception as e:
            print(f"Помилка завантаження для ID {pid}: {e}")
            
download_vhi_data()

Створено директорію: vhi_data
Завантаження з підключенням до ID 1...
Завантаження з підключенням до ID 2...
Завантаження з підключенням до ID 3...
Завантаження з підключенням до ID 4...
Завантаження з підключенням до ID 5...
Завантаження з підключенням до ID 6...
Завантаження з підключенням до ID 7...
Завантаження з підключенням до ID 8...
Завантаження з підключенням до ID 9...
Завантаження з підключенням до ID 10...
Завантаження з підключенням до ID 11...
Завантаження з підключенням до ID 12...
Завантаження з підключенням до ID 13...
Завантаження з підключенням до ID 14...
Завантаження з підключенням до ID 15...
Завантаження з підключенням до ID 16...
Завантаження з підключенням до ID 17...
Завантаження з підключенням до ID 18...
Завантаження з підключенням до ID 19...
Завантаження з підключенням до ID 20...
Завантаження з підключенням до ID 21...
Завантаження з підключенням до ID 22...
Завантаження з підключенням до ID 23...
Завантаження з підключенням до ID 24...
Завантаження з підк

### Завдання: 
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області. Реалізувати процедуру зміни індексів так, щоб області індексувалися за українською абеткою.

In [103]:
import os

directory = "vhi_data"
files = [f for f in os.listdir(directory) if f.endswith(".csv")]

if files:
    test_file = os.path.join(directory, files[0])
    print(f"Перевіряємо файл: {test_file}")
    with open(test_file, 'r', encoding='utf-8') as f:
        for i in range(5):
            print(f"Рядок {i+1}: {repr(f.readline())}")
else:
    print("У папці немає .csv файлів!")

Перевіряємо файл: vhi_data\vhi_id_10_20260531_212253.csv
Рядок 1: "Mean data for UKR  Province= 10: Khmel'nyts'kyy,  from 1981 to 2024, weekly; version='GC_current'<br>for cropland area only<br>\n"
Рядок 2: 'year,week, SMN,SMT,VCI,TCI, VHI<br>\n'
Рядок 3: '<tt><pre>1982, 1, 0.059,258.24, 51.11, 48.78, 49.95,\n'
Рядок 4: '1982, 2, 0.063,261.53, 55.89, 38.20, 47.04,\n'
Рядок 5: '1982, 3, 0.063,263.45, 57.30, 32.69, 44.99,\n'


In [ ]:
import io

def load_and_clean_data(directory=DOWNLOAD_DIR):
    all_dfs = []
    if not os.path.exists(directory):
        print(f"Директорія {directory} відсутня.")
        return pd.DataFrame()
        
    files = [f for f in os.listdir(directory) if f.startswith("vhi_id_") and f.endswith(".csv")]
    
    if not files:
        print("Файли для зчитування відсутні у папці.")
        return pd.DataFrame()

    for filename in files:
        match = re.search(r"vhi_id_(\d+)_", filename)
        if not match:
            continue
        noaa_id = int(match.group(1))
        filepath = os.path.join(directory, filename)
        
        try:
            # Читаємо файл як сирий текст
            with open(filepath, 'r', encoding='utf-8') as file:
                lines = file.readlines()
            
            clean_lines = []
            for line in lines:
                line = line.strip()
                
                # Видаляємо html-теги <tt><pre> або <br>
                line = re.sub(r'<[^>]+>', '', line)
                
                # Якщо після видалення тегів рядок закінчується комою — зрізаємо її
                if line.endswith(','):
                    line = line[:-1]
                
                # Пропускаємо заголовки (залишаємо тільки ті рядки, де є дані — починаються з цифри року)
                # re.match шукає цифру навіть якщо перед нею були пробіли, завдяки strip()
                if line and re.match(r'^\d{4}', line):
                    clean_lines.append(line)
            
            if not clean_lines:
                continue
                
            # Перетворюємо очищені текстові рядки на DataFrame
            clean_text = '\n'.join(clean_lines)
            df = pd.read_csv(
                io.StringIO(clean_text),
                header=None,
                names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'],
                sep=r'\s*,\s*',
                engine='python'
            )
            
            if df.empty:
                continue
                
            # Конвертація типів даних
            for col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                
            df = df.dropna(subset=['Year', 'Week', 'VHI'])
            df = df[df['VHI'] >= 0]
            
            df['Year'] = df['Year'].astype(int)
            df['Week'] = df['Week'].astype(int)
            
            # Створення колонки Datetime на початку
            df['Date'] = pd.to_datetime(
                df['Year'].astype(str) + '-W' + df['Week'].astype(str) + '-1', 
                format='%G-W%V-%u', 
                errors='coerce'
            )
            
            # Тимчасово записуємо назву регіону за старим ID
            if noaa_id in NOAA_TO_UKR:
                df['Region_Name'] = NOAA_TO_UKR[noaa_id]['name']
            else:
                df['Region_Name'] = "Unknown"
                
            all_dfs.append(df)
        except Exception as e:
            print(f"Помилка зчитування файлу {filename}: {e}")
        
    if not all_dfs:
        print("Не вдалося розпарсити жодного файлу. Перевірте вміст папки vhi_data.")
        return pd.DataFrame()
        
    final_df = pd.concat(all_dfs, ignore_index=True)
    
    # Нова індексація областей за українською абеткою
    unique_regions = sorted(final_df['Region_Name'].unique())
    if "Unknown" in unique_regions:
        unique_regions.remove("Unknown")
        
    alphabetical_id_map = {name: idx + 1 for idx, name in enumerate(unique_regions)}
    alphabetical_id_map["Unknown"] = 99
    
    final_df['Region_ID'] = final_df['Region_Name'].map(alphabetical_id_map)
    
    # Встановлюємо правильний порядок стовпчиків
    column_order = ['Region_ID', 'Region_Name', 'Year', 'Week', 'Date', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
    final_df = final_df[column_order]
    
    return final_df

main_df = load_and_clean_data()
print(f"Загальна кількість записів після очищення: {len(main_df)}")
display(main_df.head())

Загальна кількість записів після очищення: 59022


,Region_ID,Region_Name,Year,Week,Date,SMN,SMT,VCI,TCI,VHI
0,12,Кіровоградська,1982,1,1982-01-04,0.059,258.24,51.11,48.78,49.95
1,12,Кіровоградська,1982,2,1982-01-11,0.063,261.53,55.89,38.20,47.04
2,12,Кіровоградська,1982,3,1982-01-18,0.063,263.45,57.30,32.69,44.99
3,12,Кіровоградська,1982,4,1982-01-25,0.061,265.10,53.96,28.62,41.29
4,12,Кіровоградська,1982,5,1982-02-01,0.058,266.42,46.87,28.57,37.72


### Завдання: 
Реалізувати процедуру для формування вибірки: Ряд VHI для області за вказаний рік.

In [105]:
def get_vhi_series_by_year(df, region_id, year):
    if df.empty or 'Region_ID' not in df.columns:
        return pd.DataFrame()
    result = df[(df['Region_ID'] == region_id) & (df['Year'] == year)][['Week', 'VHI']].sort_values('Week')
    return result

# Приклад виклику для Вінницької області (ID: 1) за 2020 рік
print("Ряд VHI для Вінницької області за 2020 рік:")
# Замість print(....to_string()) робимо чистий display()
display(get_vhi_series_by_year(main_df, 1, 2020))

Ряд VHI для Вінницької області за 2020 рік:


,Week,VHI
56576,1,51.61
56577,2,52.53
56578,3,51.83
56579,4,52.64
56580,5,51.71
56581,6,48.70
56582,7,44.13
56583,8,41.16
56584,9,40.29
56585,10,40.16


### Завдання: 
Реалізувати процедуру для формування вибірки: Ряд VHI за вказаний діапазон років для вказаних областей.

In [106]:
def get_vhi_by_regions_and_years(df, region_ids, start_year, end_year):
    if df.empty or 'Region_ID' not in df.columns:
        return pd.DataFrame()
    result = df[
        (df['Region_ID'].isin(region_ids)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ][['Region_ID', 'Region_Name', 'Year', 'Week', 'VHI']].sort_values(['Region_ID', 'Year', 'Week'])
    return result

# Приклад виклику для областей з ID 1 та 2 за період 2021-2022 років
print("Ряд VHI за 2021-2022 роки для областей ID [1, 2]:")
# Замінено на display() для гарної HTML-таблиці
display(get_vhi_by_regions_and_years(main_df, [1, 2], 2021, 2022))

Ряд VHI за 2021-2022 роки для областей ID [1, 2]:


,Region_ID,Region_Name,Year,Week,VHI
56628,1,Івано-Франківська,2021,1,48.95
56629,1,Івано-Франківська,2021,2,50.81
56630,1,Івано-Франківська,2021,3,52.86
56631,1,Івано-Франківська,2021,4,56.97
56632,1,Івано-Франківська,2021,5,57.15
...,...,...,...,...,...
43611,2,Волинська,2022,48,39.35
43612,2,Волинська,2022,49,38.39
43613,2,Волинська,2022,50,38.92
43614,2,Волинська,2022,51,40.75


### Завдання: 
Реалізувати процедуру пошуку екстремумів (min та max) для вказаних областей та років, середнього, медіани.


In [107]:
def get_vhi_statistics(df, region_ids, start_year, end_year):
    if df.empty or 'Region_ID' not in df.columns:
        return "Немає даних для розрахунку або структура DataFrame некоректна."
        
    filtered_df = df[
        (df['Region_ID'].isin(region_ids)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ]
    
    if filtered_df.empty:
        return "Немає даних для розрахунку статистичних показників за вказаний період."
        
    stats = filtered_df.groupby(['Region_ID', 'Region_Name', 'Year'])['VHI'].agg(
        Мінімум='min',
        Максимум='max',
        Середнє='mean',
        Медіана='median'
    ).reset_index()
    
    return stats

# Приклад виклику для Вінницької області (ID: 1) за період 2019-2021 років
print("Статистичні показники VHI для області ID 1 за 2019-2021 роки:")
result = get_vhi_statistics(main_df, [1], 2019, 2021)

if isinstance(result, pd.DataFrame):
    display(result)
else:
    print(result)

Статистичні показники VHI для області ID 1 за 2019-2021 роки:


,Region_ID,Region_Name,Year,Мінімум,Максимум,Середнє,Медіана
0,1,Івано-Франківська,2019,24.05,56.60,41.079231,40.120
1,1,Івано-Франківська,2020,20.89,66.26,43.276923,41.055
2,1,Івано-Франківська,2021,34.71,68.30,49.377500,47.695
